# Interactive review & ATLAS export

End-to-end example: run Phecoder on a few phecodes, review the top-K retrieved ICDs in a widget, and export the curated selection as OHDSI ATLAS concept-set JSON.

Requires the optional extra: `pip install 'phecoder[review]'`.

In [1]:
from phecoder import Phecoder, load_icd_df

icd_df = load_icd_df()

# Optional: attach OMOP concept_id (+ vocabulary_id) for ATLAS export.
# Skip this merge if you only want to review/save selections.
# icd_df = icd_df.merge(my_omop_concept_map, on='icd_code', how='left')

pc = Phecoder(
    phecodes=['Major depressive disorder', 'Generalized anxiety disorder'],
    icd_df=icd_df,
    models="sentence-transformers/all-MiniLM-L12-v2", 
    output_dir='./review_demo',
    st_search_kwargs={'top_k': 50},
)
pc.run()

## Review widget

Open the widget. Each phecode has its own tab; rows above `score_threshold` are pre-checked for fast triage. Toggle checkboxes, then type a filename and click **Save selection** (or call `session.save(...)` from code).

In [2]:
session = pc.review(top_k=100, score_threshold=None, models="sentence-transformers/all-MiniLM-L12-v2")
session

In [ ]:
# Programmatic save (parquet = flat table; json = nested concept-set)
session.save('./review_demo/picks.parquet')
session.save('./review_demo/picks.json')
session.selection.head()

## ATLAS concept-set export

Requires a `concept_id` column on the selection (supplied via `icd_df` above). Writes one JSON file per phecode, or a single bundle if the path ends in `.json`. Import the result into ATLAS via *Concept Sets → Import*.

In [3]:
pc.export_atlas(session, './review_demo/atlas_concept_sets')        # directory
pc.export_atlas(session, './review_demo/atlas_concept_sets.json')   # bundle

ValueError: ATLAS export requires a 'concept_id' column. Add OMOP concept_id values to your icd_df before running Phecoder (any extra columns on icd_df are preserved end-to-end). See README section 'Interactive review & ATLAS export'.